# 7. Pilot Readiness — Is the Proposal Ready for a Pilot?

This stage does **not** run an A/B test. It checks whether the supplied
schema contains the fields required to evaluate one.

**What the readiness check covers:**
1. Whether a historical cohort matches the proposed routing rule
2. Whether a stable customer key exists for treatment/control assignment
3. Whether ticket creation time exists to start the response-time clock
4. Whether first-response duration can be calculated
5. Whether monthly product usage can be aligned before and after a ticket
6. Whether historical treatment assignment exists

**How product sessions would be used:** `sessions` is recorded once per
customer-product-month. It's meaningful as a **future secondary outcome** —
comparing post-ticket session stability between treatment and control
customers — but not usable as a current outcome, since there's no
ticket-creation timestamp to define which months are before/after a
ticket.

In [1]:
from common import ARTIFACTS
import json
import pandas as pd
from IPython.display import Markdown, display

tickets = pd.read_parquet(ARTIFACTS / "tickets.parquet")
usage = pd.read_parquet(ARTIFACTS / "usage.parquet")
ticket_view = pd.read_parquet(ARTIFACTS / "ticket_view.parquet")
with open(ARTIFACTS / "routing_results.json") as f:
    routing_results = json.load(f)
moved_count = routing_results["moved_count"]
print(f"Loaded tickets, usage, ticket_view. moved_count from stage 6: {moved_count:,}")

Loaded tickets, usage, ticket_view. moved_count from stage 6: 125


## 7.1 Check the Schema Against Pilot Requirements

In [2]:
ticket_columns = set(tickets.columns)
usage_columns = set(usage.columns)
has_customer_key = "customer_id" in ticket_view.columns and ticket_view["customer_id"].notna().all()
ticket_created_candidates = {"ticket_created_at", "ticket_creation_time", "created_at", "ticket_date"}
has_ticket_created_at = bool(ticket_columns.intersection(ticket_created_candidates))
has_first_response_timestamp = (
    "first_response_time" in ticket_columns and tickets["first_response_time"].notna().any()
)
can_measure_first_response = has_ticket_created_at and has_first_response_timestamp
has_usage_month = "month" in usage_columns
can_align_post_ticket_engagement = has_ticket_created_at and has_usage_month
treatment_candidates = {"treatment", "experiment_group", "routing_policy"}
has_historical_treatment = bool(ticket_columns.intersection(treatment_candidates))
print("Schema checks computed.")

Schema checks computed.


In [3]:
pilot_readiness = pd.DataFrame({
    "requirement": [
        "Historical tickets matching the proposed rule",
        "Customer-level randomisation key",
        "Ticket creation timestamp",
        "First-response interval",
        "Post-ticket engagement alignment",
        "Historical treatment assignment",
    ],
    "status": [
        f"Available: {moved_count:,} tickets",
        "Available" if has_customer_key else "Missing",
        "Available" if has_ticket_created_at else "Missing",
        "Measurable" if can_measure_first_response else "Blocked",
        "Measurable" if can_align_post_ticket_engagement else "Blocked",
        "Available" if has_historical_treatment else "Not present",
    ],
    "consequence": [
        "Queue workload can be simulated",
        "A future pilot can randomise by customer",
        "Required to start the response-time clock",
        "Cannot evaluate the primary service KPI",
        "Cannot define before/after usage around a ticket",
        "No historical causal comparison is possible",
    ],
})
display(pilot_readiness.style.hide(axis="index"))

impact_ready = has_customer_key and can_measure_first_response and can_align_post_ticket_engagement
readiness_message = (
    "The supplied data can evaluate the pilot."
    if impact_ready
    else "Routing workload can be simulated, but impact evaluation is blocked by missing ticket-creation timing."
)
display(Markdown(f"**Readiness result:** {readiness_message}"))

requirement,status,consequence
Historical tickets matching the proposed rule,Available: 125 tickets,Queue workload can be simulated
Customer-level randomisation key,Available,A future pilot can randomise by customer
Ticket creation timestamp,Missing,Required to start the response-time clock
First-response interval,Blocked,Cannot evaluate the primary service KPI
Post-ticket engagement alignment,Blocked,Cannot define before/after usage around a ticket
Historical treatment assignment,Not present,No historical causal comparison is possible


**Readiness result:** Routing workload can be simulated, but impact evaluation is blocked by missing ticket-creation timing.

## 7.2 How Atlassian Should Validate the Proposal

- **Treatment:** eligible Hub Premium/Enterprise Low or Medium tickets
  receive a one-level queue uplift.
- **Control:** eligible customers remain under current routing.
- **Randomisation:** assign by customer, not ticket, to prevent the same
  customer receiving both policies.
- **Primary KPI:** first-response time measured from a newly captured
  ticket-creation timestamp.
- **Secondary outcomes:** resolution time, repeat tickets, monthly
  sessions, and active days after the ticket.
- **Guardrails:** Critical-ticket delay, other-plan delay, queue workload,
  and uplift reversals.

Thirty days is a proposed starting window, not a statistically justified
duration. Atlassian should determine sample size from live eligible-ticket
arrivals and response-time variation.

**Improvement path:** collect valid timing first, run the limited pilot
second, and only then decide whether collaboration-aware routing improves
customer experience.